# 02 · Score Compounds With the R Model

This notebook adds the model's predictions to the dashboard data. It scores every compound in `CHEMBL1978_COMPOUNDS` (created by `01_prepare_training_data.ipynb`) with the R model served by `QSAR_PLUMBER_SVC`, and writes `dashboard_predictions.csv` to the app's stage.

Run it after the R service is running. The notebook calls the service over its internal Snowpark Container Services DNS name, so no token or public endpoint is needed.

## Configuration

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

DATABASE = "QSAR_DB"
SCHEMA = "PUBLIC"
SERVICE = "QSAR_PLUMBER_SVC"
STAGE = f"@{DATABASE}.{SCHEMA}.QSAR_STAGE"

session.use_database(DATABASE)
session.use_schema(SCHEMA)
print(session.get_current_database(), session.get_current_schema())

## Load Compounds

In [ ]:
import pandas as pd
import requests

compounds = session.table("CHEMBL1978_COMPOUNDS").to_pandas()   # bounded: a few thousand compounds
compounds.columns = [c.lower() for c in compounds.columns]
compounds = compounds.sort_values(["pchembl_value", "molecule_chembl_id"],
                                  ascending=[False, True]).reset_index(drop=True)
compounds["activity_label"] = compounds["activity_class"].map({1: "Active", 0: "Inactive"})
print(f"Compounds: {len(compounds)}")

## Score Compounds

Requests use the service-function row format, `[row_index, smiles]`, in batches of 50.

In [ ]:
svc = session.sql(f"SHOW SERVICES LIKE '{SERVICE}' IN SCHEMA {DATABASE}.{SCHEMA}").collect()
predict_url = f"http://{svc[0]['dns_name']}:5000/predict"
print("Service:", predict_url)

BATCH_SIZE = 50
smiles = compounds["canonical_smiles"].tolist()
predictions = [None] * len(smiles)
for start in range(0, len(smiles), BATCH_SIZE):
    batch = smiles[start:start + BATCH_SIZE]
    resp = requests.post(predict_url, json={"data": [[i, s] for i, s in enumerate(batch)]}, timeout=120)
    resp.raise_for_status()
    for idx, pic50, *_ in resp.json()["data"]:
        predictions[start + idx] = pic50

pred_df = pd.DataFrame({
    "actual_pic50":   compounds["pchembl_value"],
    "pred_pic50":     pd.to_numeric(pd.Series(predictions), errors="coerce").round(3),
    "activity_label": compounds["activity_label"],
}).dropna(subset=["pred_pic50"]).reset_index(drop=True)
print(f"Predictions: {len(pred_df)} of {len(compounds)}")

## Compare Predictions With Measured Values

Compounds near the diagonal are predicted well. These are training and test compounds together, so the fit looks better than the held-out test metrics in `training_metrics.json`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(5.5, 5))
for label, color in [("Inactive", "#00bcd4"), ("Active", "#e040fb")]:
    sub = pred_df[pred_df["activity_label"] == label]
    ax.scatter(sub["actual_pic50"], sub["pred_pic50"], s=8, alpha=0.5, color=color, label=label)
lims = [pred_df[["actual_pic50", "pred_pic50"]].min().min(), pred_df[["actual_pic50", "pred_pic50"]].max().max()]
ax.plot(lims, lims, "k--", linewidth=1)
ax.set_xlabel("Measured pIC50"); ax.set_ylabel("Predicted pIC50"); ax.legend()
plt.tight_layout()
plt.show()

rmse = np.sqrt(np.mean((pred_df["pred_pic50"] - pred_df["actual_pic50"]) ** 2))
print(f"RMSE across all compounds: {rmse:.3f}")

## Write Predictions to the Stage

In [ ]:
import os
import tempfile

path = os.path.join(tempfile.mkdtemp(), "dashboard_predictions.csv")
pred_df.to_csv(path, index=False)
session.file.put(path, STAGE, auto_compress=False, overwrite=True)
print(f"dashboard_predictions.csv {len(pred_df)} rows")

In [ ]:
%%sql -r stage_files
LIST @QSAR_STAGE PATTERN = '.*(dashboard_|chembl1978_cleaned).*'

## Next Steps

Restart the Streamlit app to pick up the new files; its data loaders cache results for an hour.